In [ ]:
import time
import pandas as pd
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys  # ต้องใช้ Keys.ENTER
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re 
from geopy.geocoders import Nominatim 
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import traceback


# BANGKOK_DISTRICTS = [
#     "สัมพันธวงศ์", "มีนบุรี", "สะพานสูง"   
# ]
# --- รายชื่อเขตข้อมูลต่ำกว่า 60 ตัว
BANGKOK_DISTRICTS = [
    "คลองสามวา", "ตลิ่งชัน", "ทวีวัฒนา", "ทุ่งครุ", "บางบอน", "พระนคร", "หนองจอก", 
    "สัมพันธวงศ์", "มีนบุรี", "สะพานสูง", "บางขุนเทียน", "ดุสิต", "ป้อมปราบศัตรูพ่าย", "หนองแขม", "หลักสี่" ,
    "วังทองหลาง", "คันนายาว", "ราษฎร์บูรณะ", "บางแค", "สายไหม", "บางกอกใหญ่", "ดอนเมือง", "จอมทอง",
]
# BANGKOK_DISTRICTS = [
#     "คลองสามวา", "ตลิ่งชัน", "ทวีวัฒนา", "ทุ่งครุ", "บางบอน", "พระนคร", "หนองจอก", 
#     "สัมพันธวงศ์", "มีนบุรี", "สะพานสูง", "บางขุนเทียน", "ดุสิต", "ป้อมปราบศัตรูพ่าย", "หนองแขม", "หลักสี่" ,
#     "วังทองหลาง", "คันนายาว", "ราษฎร์บูรณะ", "บางแค", "สายไหม", "บางกอกใหญ่", "ดอนเมือง", "จอมทอง",
#     "บางซื่อ", "ลาดพร้าว", "บางพลัด", "บางคอแหลม", "บึงกุ่ม", "บางกอกน้อย", "ภาษีเจริญ", "ธนบุรี",
#     "บางกะปิ","สาทร", "ประเวศ", "ดินแดง", "บางรัก", "ยานนาวา", "พญาไท", "ลาดกระบัง", "บางเขน" 
    
# ]
# --- รายชื่อ 50 เขต กทม. ---        
# BANGKOK_DISTRICTS = [
#     "พระนคร", "ดุสิต", "หนองจอก", "บางรัก", "บางเขน", "บางกะปิ", "ปทุมวัน", "ป้อมปราบศัตรูพ่าย", 
#     "พระโขนง", "มีนบุรี", "ลาดกระบัง", "ยานนาวา", "สัมพันธวงศ์", "พญาไท", "ธนบุรี", "บางกอกใหญ่", 
#     "ห้วยขวาง", "คลองสาน", "ตลิ่งชัน", "บางกอกน้อย", "บางขุนเทียน", "ภาษีเจริญ", "หนองแขม", "ราษฎร์บูรณะ", 
#     "บางพลัด", "ดินแดง", "บึงกุ่ม", "สาทร", "บางซื่อ", "จตุจักร", "บางคอแหลม", "ประเวศ", "คลองเตย", 
#     "สวนหลวง", "จอมทอง", "ดอนเมือง", "ราชเทวี", "ลาดพร้าว", "วัฒนา", "บางแค", "หลักสี่", "สายไหม", 
#     "คันนายาว", "สะพานสูง", "วังทองหลาง", "คลองสามวา", "บางนา", "ทวีวัฒนา", "ทุ่งครุ", "บางบอน"
# ]# --- Helper Functions ---

def get_property_value(soup, keyword):

    property_groups = soup.find_all("div", class_="detail-list-property")

    for group in property_groups:

        property_items = group.find_all("div", class_="detail-col-property-list")

        for item in property_items:

            title_span = item.find("span", class_="detail-property-list-title")

            if title_span and keyword in title_span.get_text(strip=True):

                value_span = item.find("span", class_="detail-property-list-text")

                if value_span:

                    return value_span.get_text(strip=True)

    return None



def extract_coords_from_google_maps_url(url):

    match = re.search(r"@(-?\d+\.\d+),(-?\d+\.\d+)", url)

    if match:

        return f"{match.group(1)},{match.group(2)}"

    return None



def reverse_geocode_coords(coords):

    if not coords: return None, None, None, None, None

    try: latitude, longitude = map(str.strip, coords.split(','))

    except ValueError: return None, None, None, None, None

    

    geolocator = Nominatim(user_agent="living_insider_scraper_team_x", timeout=10) # เปลี่ยน User Agent นิดหน่อยกันเหนียว

    try:

        location = geolocator.reverse((latitude, longitude), exactly_one=True, language='th')

        if location and location.raw and 'address' in location.raw:

            full_address_geopy = location.address # ค่านี้คือที่อยู่เต็มจาก Geopy

            address_parts = location.raw['address']

            return address_parts.get('quarter'), address_parts.get('suburb'), address_parts.get('city'), address_parts.get('postcode'), full_address_geopy

    except: pass

    return None, None, None, None, None



def init_driver():

    options = uc.ChromeOptions()

    options.add_argument('--window-size=1280,960')

    options.add_argument('--disable-popup-blocking')

    options.add_argument('--no-sandbox')

    options.add_argument('--disable-dev-shm-usage')

    driver = uc.Chrome(options=options)

    return driver



# --- MAIN SCRAPER ---

def scrape_living_insider():

    PAGES_PER_DISTRICT = 2 

    driver = init_driver()

    all_data_list = []



    print(f"🚀 เริ่มต้น Scraper รายเขต ({len(BANGKOK_DISTRICTS)} เขต) [Logic: Geopy Address if Map Text Exists]")



    for dist_idx, district_name in enumerate(BANGKOK_DISTRICTS):

        print(f"\n════════════════════════════════════════════════════════")

        print(f"📍 เขต: {district_name} ({dist_idx+1}/{len(BANGKOK_DISTRICTS)})")

        print(f"════════════════════════════════════════════════════════")



        # --- Loop Pages by URL Construction ---

        for page in range(1, PAGES_PER_DISTRICT + 1):

            

            target_url = f"https://www.livinginsider.com/searchword/Condo/Buysell/{page}/{district_name}.html"

            print(f"   📄 Page {page}/{PAGES_PER_DISTRICT} : Accessing URL...")

            time.sleep(3) 

            try:

                driver.get(target_url)

                try:

                    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "item-desc")))

                except:

                    print(f"⚠️ ไม่พบข้อมูลในหน้านี้ (อาจหมดแล้ว)")

                    break 



                soup_main = BeautifulSoup(driver.page_source, 'html.parser')

                items = soup_main.find_all('div', class_='item-desc')

                

                unique_links = set()

                skipped_count = 0



                # --- Filter Pinned Posts ---

                for item in items:

                    is_pinned = item.find("rect", {"fill": "white", "rx": "11.75"})

                    if is_pinned:

                        skipped_count += 1

                        continue 



                    a_tag = item.find_parent('a') 

                    if a_tag and 'href' in a_tag.attrs: unique_links.add(a_tag['href'])

                    a_tag_inner = item.find('a')

                    if a_tag_inner and 'href' in a_tag_inner.attrs: unique_links.add(a_tag_inner['href'])

                

                all_links = list(unique_links)

                print(f"      🔎 เจอ {len(all_links)} รายการ (ข้ามปักหมุด {skipped_count})")



                # --- Loop Items ---

                for i, link in enumerate(all_links): 

                    full_url = link if link.startswith("http") else f"https://www.livinginsider.com{link}"

                    print(f"      [{i+1}/{len(all_links)}] Scraping... ", end="")

                    

                    # Reset Variables

                    coords = None

                    sub_district = None; district = None; province = None; postcode = None

                    full_address = None # เริ่มต้นเป็น None ไว้ก่อน

                    

                    try:

                        driver.get(full_url)

                        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "text_project_detail_green")))

                        

                        # --- 🔥 MAP LOGIC (UPDATED) 🔥 ---

                        try:

                            map_link = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.CLASS_NAME, "detail-view-map")))

                            driver.get(map_link.get_attribute("href"))

                            time.sleep(5) 

                            

                            # 1. ดึงพิกัด

                            coords = extract_coords_from_google_maps_url(driver.current_url)

                            

                            # 2. เช็คว่ามี Text Address ใน Google Map หรือไม่ (ตาม Class ที่ระบุ)

                            soup_map = BeautifulSoup(driver.page_source, 'html.parser')

                            address_div = soup_map.find("div", class_="Io6YTe fontBodyMedium kR99db fdkmkc") or soup_map.find("div", class_="fontBodyMedium")

                            

                            # 3. เงื่อนไขการใส่ค่า

                            if address_div:

                                # ถ้าเจอ Div นี้ -> ให้ไปเรียก Geopy

                                if coords: 

                                    sub_district, district, province, postcode , geopy_addr = reverse_geocode_coords(coords)

                                    # เอาค่าจาก Geopy ใส่ full_address

                                    full_address = geopy_addr 

                            else:

                                # ถ้าไม่เจอ Div นี้ -> full_address เป็น None (ไม่ต้องทำอะไรเพิ่มเพราะ reset ไว้แล้ว)

                                pass



                            # กลับหน้าเดิม

                            driver.get(full_url)

                            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "text_project_detail_green")))

                        except Exception as e: 

                            # กรณี Map Error หรือหาไม่เจอ ให้กลับหน้าเดิม

                            try: driver.get(full_url)

                            except: pass



                        # --- Content Logic ---

                        soup = BeautifulSoup(driver.page_source, 'html.parser')

                        title_div = soup.find("span", class_="text_project_detail_green")

                        date_span = soup.find("span", class_="lv-small-font grey font_10_date font_sarabun")

                        

                        try:

                            more_btn = WebDriverWait(driver, 3).until(EC.element_to_be_clickable((By.XPATH, "//span[contains(@class, 'font_title_more')]//span[contains(text(), 'ข้อมูลเพิ่มเติม')]")))

                            driver.execute_script("arguments[0].click();", more_btn)

                            time.sleep(0.5)

                            soup = BeautifulSoup(driver.page_source, 'html.parser')

                        except: pass 



                        price = None; price_sqm = None

                        price_div = soup.find("div", class_="box_price_mb")

                        if price_div:

                            if price_div.find("span", class_="price-detail"): price = price_div.find("span", class_="price-detail").get_text(strip=True)

                            if price_div.find("span", class_="price_cal_area_text_modal"): price_sqm = price_div.find("span", class_="price_cal_area_text_modal").get_text(strip=True)



                        row_data = {

                            "url": full_url,

                            "title": title_div.get_text(strip=True) if title_div else None,

                            "publish_date": date_span.get_text(strip=True).replace("สร้างเมื่อ", "").replace("ปรับปรุง", "").strip() if date_span else None, 

                            "price": price,

                            "price_per_sqm": price_sqm,

                            "usable_area": get_property_value(soup, "พื้นที่ใช้สอย"),

                            "floor": get_property_value(soup, "ชั้น"),

                            "bedroom": get_property_value(soup, "ห้องนอน"),

                            "restroom": get_property_value(soup, "ห้องน้ำ"),

                            "coords": coords,   

                            "full_address": full_address, # ค่านี้มาจาก Geopy เฉพาะตอนที่เจอ Address Div เท่านั้น

                            "sub_district": sub_district, 

                            "district": district,    

                            "province": province,    

                            "postcode": postcode,    

                        }

                        all_data_list.append(row_data)

                        print("✅ Done")



                    except Exception as e:

                        if "invalid session" in str(e):

                            driver.quit(); driver = init_driver()

                        else: print("❌ Skip"); continue

            

            except Exception as e:

                if "invalid session" in str(e): driver = init_driver()

                continue

            

            if len(all_data_list) > 0:

                pd.DataFrame(all_data_list).to_csv("living_insider_full_data_districts.csv", index=False, encoding="utf-8-sig")

                print(f"💾 Saved Total: {len(all_data_list)} records")



    driver.quit()

    print(f"\n🎉 Completed! Total: {len(all_data_list)}")



if __name__ == "__main__":

    scrape_living_insider()

🚀 เริ่มต้น Scraper รายเขต (3 เขต) [Logic: Geopy Address if Map Text Exists]

════════════════════════════════════════════════════════
📍 เขต: สัมพันธวงศ์ (1/3)
════════════════════════════════════════════════════════
   📄 Page 1/2 : Accessing URL...
      🔎 เจอ 19 รายการ (ข้ามปักหมุด 0)
      [1/19] Scraping... ✅ Done
      [2/19] Scraping... ✅ Done
      [3/19] Scraping... ✅ Done
💾 Saved Total: 3 records
   📄 Page 2/2 : Accessing URL...
⚠️ ไม่พบข้อมูลในหน้านี้ (อาจหมดแล้ว)

════════════════════════════════════════════════════════
📍 เขต: มีนบุรี (2/3)
════════════════════════════════════════════════════════
   📄 Page 1/2 : Accessing URL...
      🔎 เจอ 48 รายการ (ข้ามปักหมุด 0)
      [1/48] Scraping... ✅ Done
      [2/48] Scraping... ✅ Done
      [3/48] Scraping... ✅ Done
💾 Saved Total: 6 records
   📄 Page 2/2 : Accessing URL...
      🔎 เจอ 48 รายการ (ข้ามปักหมุด 0)
      [1/48] Scraping... ✅ Done
      [2/48] Scraping... ✅ Done
      [3/48] Scraping... ✅ Done
💾 Saved Total: 9 records

═

In [1]:
import pandas as pd
# from datetime import datetime, timedelta
# import re
# import numpy as np

# อ่านไฟล์ (แก้ไขชื่อไฟล์ตามที่คุณใช้จริง ถ้าใช้ชื่อเดิมก็ไม่ต้องแก้ครับ)
df = pd.read_csv("ddproperty_cleaned_ver4_prefix.csv")

# นับจำนวนแถวของแต่ละเขต
district_counts = df['district'].value_counts()

# แสดงรายชื่อเขตและจำนวนข้อมูล
print(district_counts)
print("-" * 30) # ขีดเส้นคั่นให้อ่านง่าย

# --- ส่วนที่เพิ่ม: นับว่ามีทั้งหมดกี่เขต ---
total_districts = len(district_counts)
print(f"มีข้อมูลทั้งหมด: {total_districts} เขต")

district
เขตปทุมวัน              202
เขตวัฒนา                202
เขตห้วยขวาง             201
เขตราชเทวี              201
เขตบางรัก               201
เขตคลองเตย              200
เขตคลองสาน              200
เขตลาดกระบัง            200
เขตจตุจักร              200
เขตบึงกุ่ม              200
เขตสาทร                 200
เขตลาดพร้าว             200
เขตคันนายาว             200
เขตจอมทอง               199
เขตสวนหลวง              199
เขตวังทองหลาง           199
เขตบางกะปิ              198
เขตประเวศ               198
เขตบางกอกน้อย           198
เขตราษฎร์บูรณะ          198
เขตยานนาวา              198
เขตภาษีเจริญ            198
เขตบางเขน               197
เขตบางพลัด              196
เขตบางคอแหลม            196
เขตพระโขนง              196
เขตธนบุรี               196
เขตดินแดง               196
เขตบางนา                196
เขตพญาไท                196
เขตบางซื่อ              195
เขตบางกอกใหญ่           189
เขตบางแค                187
เขตหลักสี่              150
เขตดอนเมือง             119
เขตสายไหม  